# Export Train-Pool Parquets for Colab

This temporary notebook loads `data/complexity_splits.pkl` created by `complexity/code.ipynb` and writes the precomputed `train_pool` split as three parquet files:
- `data/train_pool_train.parquet`
- `data/train_pool_val.parquet`
- `data/train_pool_test.parquet`

Run it locally in the environment where `complexity_splits.pkl` opens correctly.


In [1]:
import pickle
from pathlib import Path

import pandas as pd
from IPython.display import display

CFG = {
    "split_artifact_path": Path("data/complexity_splits.pkl"),
    "export_dir": Path("data"),
    "train_pool_key": "train_pool",
    "text_col": "text",
    "label_col": "label",
    "encoded_label_col": "label_enc",
}

CFG["export_dir"].mkdir(parents=True, exist_ok=True)
print(f"Split artifact: {CFG['split_artifact_path']}")
print(f"Export dir: {CFG['export_dir']}")


Split artifact: data/complexity_splits.pkl
Export dir: data


In [2]:
if not CFG["split_artifact_path"].exists():
    raise FileNotFoundError(
        f"Missing split artifact: {CFG['split_artifact_path']}\n"
        "Run complexity/code.ipynb data preparation first or copy the pickle into data/."
    )

with open(CFG["split_artifact_path"], "rb") as f:
    split_artifacts = pickle.load(f)

splits = split_artifacts["splits"]
train_pool_split = splits[CFG["train_pool_key"]]
print(f"Loaded split keys: {sorted(train_pool_split.keys())}")


Loaded split keys: ['X_test', 'X_test_raw', 'X_train', 'X_train_raw', 'X_val', 'X_val_raw', 'df_test', 'df_train', 'df_val', 'y_test', 'y_train', 'y_val']


In [3]:
CLASSES = ["A1-A2", "B1-B2", "C1-C2"]
id2class = {i: c for i, c in enumerate(CLASSES)}


def export_frame(split_dict: dict, split_name: str) -> dict:
    df_key = f"df_{split_name}"
    if df_key in split_dict:
        frame = split_dict[df_key].copy().reset_index(drop=True)
    else:
        raw_key = f"X_{split_name}_raw"
        clean_key = f"X_{split_name}"
        target_key = f"y_{split_name}"
        missing = [k for k in [raw_key, clean_key, target_key] if k not in split_dict]
        if missing:
            raise ValueError(f"Cannot reconstruct {split_name}: missing keys {missing}")

        frame = pd.DataFrame({
            "text_raw": split_dict[raw_key],
            "text_clean": split_dict[clean_key],
            CFG["encoded_label_col"]: split_dict[target_key],
        })
        frame[CFG["text_col"]] = frame["text_raw"]
        frame[CFG["label_col"]] = frame[CFG["encoded_label_col"]].map(id2class)

    if CFG["text_col"] not in frame.columns and "text_raw" in frame.columns:
        frame[CFG["text_col"]] = frame["text_raw"]
    if CFG["label_col"] not in frame.columns and CFG["encoded_label_col"] in frame.columns:
        frame[CFG["label_col"]] = frame[CFG["encoded_label_col"]].map(id2class)
    if CFG["encoded_label_col"] not in frame.columns:
        raise ValueError(f"{split_name}: missing {CFG['encoded_label_col']} column after export preparation")

    frame[CFG["encoded_label_col"]] = pd.to_numeric(frame[CFG["encoded_label_col"]], errors="raise").astype("int64")
    out_path = CFG["export_dir"] / f"{CFG['train_pool_key']}_{split_name}.parquet"
    frame.to_parquet(out_path, index=False)

    label_counts = frame[CFG["encoded_label_col"]].value_counts().sort_index().to_dict()
    label_counts = {id2class.get(int(k), str(k)): int(v) for k, v in label_counts.items()}
    return {
        "split": split_name,
        "rows": int(len(frame)),
        "labels": label_counts,
        "columns": ", ".join(frame.columns),
        "path": str(out_path),
    }


export_rows = [export_frame(train_pool_split, split_name) for split_name in ["train", "val", "test"]]
export_summary_df = pd.DataFrame(export_rows)
display(export_summary_df)


,split,rows,labels,columns,path
0,train,31305,"{'A1-A2': 9518, 'B1-B2': 17056, 'C1-C2': 4731}","text, label, source_dataset, text_len, text_ra...",data/train_pool_train.parquet
1,val,6709,"{'A1-A2': 2040, 'B1-B2': 3655, 'C1-C2': 1014}","text, label, source_dataset, text_len, text_ra...",data/train_pool_val.parquet
2,test,6709,"{'A1-A2': 2040, 'B1-B2': 3655, 'C1-C2': 1014}","text, label, source_dataset, text_len, text_ra...",data/train_pool_test.parquet


In [4]:
for split_name in ["train", "val", "test"]:
    path = CFG["export_dir"] / f"{CFG['train_pool_key']}_{split_name}.parquet"
    df = pd.read_parquet(path)
    print(f"{path.name}: {df.shape} | columns={df.columns.tolist()}")


train_pool_train.parquet: (31305, 7) | columns=['text', 'label', 'source_dataset', 'text_len', 'text_raw', 'text_clean', 'label_enc']
train_pool_val.parquet: (6709, 7) | columns=['text', 'label', 'source_dataset', 'text_len', 'text_raw', 'text_clean', 'label_enc']
train_pool_test.parquet: (6709, 7) | columns=['text', 'label', 'source_dataset', 'text_len', 'text_raw', 'text_clean', 'label_enc']
